In [2]:
import os
import re

import pandas as pd

province_dict = {
    "Anhui": "安徽省",
    "Beijing": "北京市",
    "Chongqin": "重庆市",
    "Chongqing": "重庆市",
    "Fujian": "福建省",
    "Guangdong": "广东省",
    "Guangxi": "广西壮族自治区",
    "Guizhou": "贵州省",
    "Hainan": "海南省",
    "Hebei": "河北省",
    "Heilongjiang": "黑龙江省",
    "Hubei": "湖北省",
    "Hunan": "湖南省",
    "Jiangsu": "江苏省",
    "Jiangxi": "江西省",
    "Liaoning": "辽宁省",
    "NeiMenggu": "内蒙古自治区",
    "Ningxia": "宁夏回族自治区",
    "Shandong": "山东省",
    "Shanghai": "上海市",
    "Shanxi": "山西省",
    "Sichuan": "四川省",
    "Tianjing": "天津市",
    "Tianjin": "天津市",
    "Xinjiang": "新疆维吾尔自治区",
    "Zhejiang": "浙江省"
}

def print_folders_in_directory(directory):
    try:
        # 遍历目录中的所有项目
        for item in os.listdir(directory):
            item_path = os.path.join(directory, item)
            # 检查项目是否为文件夹
            if os.path.isdir(item_path):
                print(item)
    except Exception as e:
        print(f"An error occurred: {e}")
        
def print_xlsx_in_directory(directory):
    try:
        # 遍历目录中的所有项目
        for root, dirs, files in os.walk(directory):
            for filename in files:
                if filename.endswith(".xlsx"):
                    print(filename)
    except Exception as e:
        print(f"An error occurred: {e}")

In [3]:
def rename_provinces_files_in_directory(directory, province_dict):
    """
    省份：重命名目录中的文件，将文件名中的拼音部分替换为对应的官方名称
    :param directory: 
    :param province_dict: 
    :return: 
    """
    try:
        # 遍历目录中的所有项目
        for item in os.listdir(directory):
            item_path = os.path.join(directory, item)
            # 检查项目是否为文件
            if os.path.isfile(item_path):
                # 获取文件名的拼音部分
                pinyin_name = item.split('_')[0]
                # 根据拼音名称查找对应的官方名称
                if pinyin_name in province_dict:
                    official_name = province_dict[pinyin_name]
                    new_item_name = f"{official_name}_数据开放.xlsx"
                    new_item_path = os.path.join(directory, new_item_name)
                    # 重命名文件
                    os.rename(item_path, new_item_path)
                    print(f"Renamed '{item}' to '{new_item_name}'")
                else:
                    print(f"No mapping found for file '{item}'")
    except Exception as e:
        print(f"An error occurred: {e}")
        
def rename_cities_files_in_directory(directory):
    """
    城市：重命名目录中的文件，将文件名中的拼音部分替换为对应的官方名称
    :param directory: 
    :param province_dict: 
    :return: 
    """
    try:
        # 遍历目录中的所有项目
        for root, dirs, files in os.walk(directory):
            for file in files:
                item_path = os.path.join(root, file)
                # 检查项目是否为文件
                if os.path.isfile(item_path):
                    # 获取文件名的拼音部分
                    pinyin_name = file.split('_')[0]
                   
                    new_item_name = f"{pinyin_name}_数据开放.xlsx"
                    new_item_path = os.path.join(root, new_item_name)
                    # 重命名文件
                    os.rename(item_path, new_item_path)
                    print(f"Renamed '{file}' to '{new_item_name}'")
            for dir in dirs:
                rename_cities_files_in_directory(os.path.join(directory, dir))
    except Exception as e:
        print(f"An error occurred: {e}")
        
def add_location_column_to_files(directory):
    try:
        # 遍历目录中的所有项目
        for item in os.listdir(directory):
            if item.endswith('数据开放.xlsx'):
                item_path = os.path.join(directory, item)
                # 读取Excel文件
                df = pd.read_excel(item_path)
                # 获取文件名的拼音部分
                pinyin_name = item.split('_')[0]
                # 检查 'location' 列是否存在
                if 'location' in df.columns:
                    print(f"File '{item}' already has 'location' column")
                    df['location'] = pinyin_name
                else:
                    # 新增列到第一列
                    df.insert(0, 'location', pinyin_name)
                    print(f"Added 'location' column to '{item}'")
                # 保存修改后的文件
                df.to_excel(item_path, index=False)

    except Exception as e:
        print(f"An error occurred: {e}")
        
def correct_location_column_in_files(directory):
    try:
        # 遍历目录中的所有项目
        for root, dirs, files in os.walk(directory):
            for file in files:
                item_path = os.path.join(root, file)
                # 检查项目是否为文件
                if os.path.isfile(item_path) and file.endswith('数据开放.xlsx'):
                    # 获取文件名的拼音部分
                    pinyin_name = file.split('_')[0]
                   
                    df = pd.read_excel(item_path)
                    # 检查 'location' 列是否存在
                    if 'location' in df.columns and df['location'][0]!= pinyin_name:
                        df['location'] = pinyin_name
                        print(f"Corrected 'location' column in '{file}'")
                        # 保存修改后的文件
                        df.to_excel(item_path, index=False)
                    elif df['location'][0] == pinyin_name:
                        print(f"File '{file}' already has correct 'location' column")
                    else:
                        print(f"No 'location' column in '{file}'")
                        
            for dir in dirs:
                correct_location_column_in_files(os.path.join(directory, dir))
    except Exception as e:
        print(f"An error occurred: {e}")
        
def merge_xlsx_files(source_directory, target_directory):
    try:
        # 遍历源目录中的所有子文件夹
        for root, dirs, files in os.walk(source_directory):
            for dir_name in dirs:
                dir_path = os.path.join(root, dir_name)
                merged_data = {}
                # 遍历子文件夹中的所有xlsx文件
                for file_name in os.listdir(dir_path):
                    if file_name.endswith('.xlsx'):
                        file_path = os.path.join(dir_path, file_name)
                        # 读取xlsx文件
                        df = pd.read_excel(file_path)
                         # 获取文件名中的“xx市”部分
                        match = re.search(r'([^市]+市)', file_name)
                        if match:
                            prefix = match.group(1)
                            if prefix not in merged_data:
                                merged_data[prefix] = []
                            merged_data[prefix].append(df)
                
                # 合并具有相同前缀的文件
                for prefix, dfs in merged_data.items():
                    if len(dfs) > 1:
                        merged_df = pd.concat(dfs, ignore_index=True)
                        # 创建目标文件夹路径
                        target_dir_path = os.path.join(target_directory, dir_name)
                        os.makedirs(target_dir_path, exist_ok=True)
                        # 保存合并后的文件
                        merged_file_path = os.path.join(target_dir_path, f"{prefix}_数据开放.xlsx")
                        merged_df.to_excel(merged_file_path, index=False)
                        print(f"Merged files with prefix '{prefix}' in folder '{dir_name}'")
    except Exception as e:
        print(f"An error occurred: {e}")

In [4]:

# 示例用法
citis_path = r'E:\pythonProject\outsource\China_province_opened_data\output\cities_output_files'
# print_folders_in_directory(directory_path)
provinces_path = r'E:\pythonProject\outsource\China_province_opened_data\output\provinces_output_files'
targeted_path = r'E:\pythonProject\outsource\China_province_opened_data\temp'
# print_xlsx_in_directory(provinces_path)
# rename_files_in_directory(provinces_path, province_dict)

# add_location_column_to_files(provinces_path)
# merge_xlsx_files(citis_path, targeted_path)
# rename_cities_files_in_directory(citis_path)
correct_location_column_in_files(citis_path)

File '上海市_数据开放.xlsx' already has correct 'location' column
File '乌兰察布市_数据开放.xlsx' already has correct 'location' column
File '呼和浩特市_数据开放.xlsx' already has correct 'location' column
File '北京市_数据开放.xlsx' already has correct 'location' column
File '乐山市_数据开放.xlsx' already has correct 'location' column
File '南充市_数据开放.xlsx' already has correct 'location' column
File '巴中市_数据开放.xlsx' already has correct 'location' column
File '广元市_数据开放.xlsx' already has correct 'location' column
File '广安市_数据开放.xlsx' already has correct 'location' column
File '德阳市_数据开放.xlsx' already has correct 'location' column
File '成都市_数据开放.xlsx' already has correct 'location' column
File '泸州市_数据开放.xlsx' already has correct 'location' column
File '眉山市_数据开放.xlsx' already has correct 'location' column
File '绵阳市_数据开放.xlsx' already has correct 'location' column
File '自贡市_数据开放.xlsx' already has correct 'location' column
File '资阳市_数据开放.xlsx' already has correct 'location' column
File '达州市_数据开放.xlsx' already has correct 'location' 

In [4]:

def rename_folders_in_directory(directory, province_dict):
    try:
        # 遍历目录中的所有项目
        for item in os.listdir(directory):
            item_path = os.path.join(directory, item)
            # 检查项目是否为文件夹
            if os.path.isdir(item_path):
                # 获取文件夹的拼音名称
                pinyin_name = item
                # 根据拼音名称查找对应的官方名称
                if pinyin_name in province_dict:
                    official_name = province_dict[pinyin_name]
                    new_item_path = os.path.join(directory, official_name)
                    # 重命名文件夹
                    os.rename(item_path, new_item_path)
                    print(f"Renamed '{item}' to '{official_name}'")
                else:
                    print(f"No mapping found for folder '{item}'")
    except Exception as e:
        print(f"An error occurred: {e}")


# 示例用法
directory_path = r'E:\pythonProject\outsource\China_province_opened_data\output\cities_output_files'
rename_folders_in_directory(directory_path, province_dict)


Renamed 'Anhui' to '安徽省'
Renamed 'Beijing' to '北京市'
Renamed 'Chongqin' to '重庆市'
Renamed 'Fujian' to '福建省'
Renamed 'Guangdong' to '广东省'
Renamed 'Guangxi' to '广西壮族自治区'
Renamed 'Guizhou' to '贵州省'
Renamed 'Hainan' to '海南省'
Renamed 'Hebei' to '河北省'
Renamed 'Heilongjiang' to '黑龙江省'
Renamed 'Hubei' to '湖北省'
Renamed 'Hunan' to '湖南省'
Renamed 'Jiangsu' to '江苏省'
Renamed 'Jiangxi' to '江西省'
Renamed 'Liaoning' to '辽宁省'
Renamed 'NeiMenggu' to '内蒙古自治区'
Renamed 'Ningxia' to '宁夏回族自治区'
Renamed 'Shandong' to '山东省'
Renamed 'Shanghai' to '上海市'
Renamed 'Shanxi' to '山西省'
Renamed 'Sichuan' to '四川省'
Renamed 'Tianjing' to '天津市'
Renamed 'Xinjiang' to '新疆维吾尔自治区'
Renamed 'Zhejiang' to '浙江省'
